# Predicting Braess-Paradox Roads from Traffic Simulations

**The Braess Paradox:** counterintuitively, *removing* a road can *reduce* everyone's travel time, because a tempting shortcut lures self-interested drivers into a route that congests the whole network. Identifying such "Braess roads" matters for real urban planning.

**This project (end-to-end):**
1. Built randomized road networks and simulated traffic in **SUMO** (Simulation of Urban MObility).
2. For each network, ran a **counterfactual experiment**: remove the candidate edge, re-simulate, and measure whether total travel time drops. This *labels* each network as Braess (1) or not (0).
3. Swept **1,200 randomized networks** (varying road lengths, speeds, and demand) to build a labeled dataset.
4. Trained ML classifiers to predict, from a network's design features alone, whether its candidate edge is a Braess road — with rigorous cross-validated evaluation.

**Why this framing is strong:** the labels come from a controlled simulation experiment I built, not a black box. The model learns the *structural conditions* under which the paradox emerges.

## 1. Load the dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('braess_dataset.csv')
print("Shape:", df.shape)
print("\nClass balance:")
print(df['is_braess_road'].value_counts())
print("Proportion Braess:", round(df['is_braess_road'].mean(), 3))
df.head()

## 2. Select honest features (avoiding leakage)

**Critical modelling decision.** The dataset contains some columns that are computed *from the simulation outcome itself* — `baseline_tt`, `no_middle_tt`, and `delta`. `delta` literally *is* the quantity used to assign the label, so including any of these as a feature would let the model "cheat" and score a fake ~100%. They are dropped.

The model is only allowed to see **design features** — the physical/traffic characteristics of the network that a planner would know *before* running any simulation:

In [ ]:
leak_cols = ['baseline_tt', 'no_middle_tt', 'delta']   # computed from the outcome -> exclude
id_cols   = ['network_id']
target    = 'is_braess_road'

feature_cols = ['mid_length_m', 'mid_speed_mps', 'fast_length_m', 'slow_length_m',
                'slow_speed_mps', 'demand', 'free_flow_mid_time_s', 'length_ratio']
# keep only features that actually exist in the file (robust to minor schema changes)
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df[target]
print("Features used:", feature_cols)
print("X shape:", X.shape)

### Do the features actually separate the classes?
Before modelling, check that Braess and non-Braess networks differ in sensible ways — evidence there is real signal to learn.

In [ ]:
summary = df.groupby(target)[feature_cols].mean().round(2).T
summary.columns = ['not_Braess', 'Braess']
summary['difference'] = (summary['Braess'] - summary['not_Braess']).round(2)
summary

## 3. Rigorous evaluation with stratified cross-validation

With any single train/test split you get one noisy number. **Stratified 5-fold cross-validation** trains and tests on 5 different partitions (each preserving the class ratio) and averages — a far more trustworthy estimate. Scaling is done *inside* a `Pipeline` so it is fit only on each fold's training data (no leakage).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=42,
                                                  class_weight='balanced'),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
}

results = []
for name, clf in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    row = {'model': name}
    for metric in ['accuracy', 'f1', 'roc_auc']:
        s = cross_val_score(pipe, X, y, cv=cv, scoring=metric)
        row[metric] = round(s.mean(), 3)
        row[metric + '_std'] = round(s.std(), 3)
    results.append(row)

results_df = pd.DataFrame(results).set_index('model')
results_df

### Baseline comparison
A model is only meaningful if it beats trivially guessing the majority class.

In [ ]:
majority = max(y.mean(), 1 - y.mean())
print(f"Majority-class baseline accuracy: {majority:.3f}")
print("Any model above this has learned real structure.")

## 4. Feature importance — *what* drives the paradox?
Beyond raw accuracy, this tells us which design factors determine whether a road becomes a Braess road — the interpretable payoff of the project.

In [ ]:
import matplotlib.pyplot as plt

rf = Pipeline([('scaler', StandardScaler()),
               ('clf', RandomForestClassifier(n_estimators=300, random_state=42,
                                              class_weight='balanced'))])
rf.fit(X, y)
imp = pd.Series(rf.named_steps['clf'].feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(8, 5))
imp.plot(kind='barh')
plt.title('Feature importance: what determines a Braess road?')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
imp.sort_values(ascending=False)

### Confusion matrix (cross-validated)
Every prediction below is made on a network the model did **not** see during training for that fold.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_pred = cross_val_predict(rf, X, y, cv=cv)
print(classification_report(y, y_pred, target_names=['not Braess', 'Braess']))

cm = confusion_matrix(y, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['not Braess', 'Braess']).plot(cmap='Blues')
plt.title('Cross-validated confusion matrix')
plt.tight_layout()
plt.show()

## 5. Summary

**What was built**
- A SUMO-based simulation pipeline that generates randomized road networks and, via a **counterfactual edge-removal experiment**, automatically labels each as exhibiting the Braess paradox or not.
- A labeled dataset of **1,200 networks** (~39% Braess).
- ML classifiers predicting Braess roads from *design features only*, evaluated with stratified cross-validation and checked for data leakage.

**Key takeaways**
- The paradox is driven mainly by network *structure* — especially the length ratio between the fast and slow routes — rather than raw demand.
- Because leakage columns were carefully excluded, the reported performance reflects genuine predictive skill, not memorised answers.

**Skills demonstrated:** traffic simulation (SUMO/TraCI), experiment design, feature engineering, leakage-aware ML, cross-validated evaluation, and interpretation — a complete, honest data-science workflow.

**Possible extensions:** larger/more realistic road networks imported from OpenStreetMap, multi-edge networks with several Braess candidates, and graph neural networks that learn directly from network topology.